# 00 · Colab runner
Thin wrapper: mounts Drive, installs the package, and calls the command-line scripts.
**All logic lives in `src/mmrdr/` and `scripts/`**, so every result can be reproduced without this notebook.

Runtime: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
# ---- paths (edit if your Drive layout differs) ----
DRIVE = "/content/drive/MyDrive/mmrdr"
REPO  = "/content/mmrdr-xai"                     # code location on the VM
REPO_ZIP = f"{DRIVE}/mmrdr-xai.zip"              # used if you have not pushed to GitHub yet
GITHUB = ""                                      # e.g. "https://github.com/hafsaabid416/mmrdr-xai.git"
DATA  = "/content/data/MMRDR-CFP"
CACHE = "/content/cache512"
RUNS  = f"{DRIVE}/runs"                          # checkpoints + logs persist on Drive

from google.colab import drive; drive.mount("/content/drive")
import os, subprocess
if not os.path.exists(REPO):
    if GITHUB: subprocess.run(["git", "clone", GITHUB, REPO], check=True)
    else:      subprocess.run(["unzip", "-q", REPO_ZIP, "-d", "/content"], check=True)
os.chdir(REPO)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -r requirements.txt
!python -m pytest -q tests

## 1 · Data on local disk (fast reads)

In [ ]:
if not os.path.exists(f"{DATA}/FP.csv"):
    !mkdir -p /content/data && cp -r "{DRIVE}/MMRDR-CFP" /content/data/
!ls {DATA}/img | wc -l

## 2 · Frozen leakage-safe split (run once; `data/splits.csv` is committed to the repo)

In [ ]:
if not os.path.exists("data/splits.csv"):
    !python scripts/make_splits.py --eda "{DRIVE}/eda/cfp_eda.csv" --out data/splits.csv
!cat data/splits_summary.json

## 3 · Preprocessed 512 px image cache (≈5–10 min first time, then restored from Drive in ≈1 min)

In [ ]:
TAR = f"{DRIVE}/cache512.tar"
if not os.path.exists(CACHE):
    if os.path.exists(TAR):
        !mkdir -p {CACHE} && tar -xf "{TAR}" -C {CACHE}
    else:
        !python scripts/prepare_images.py --src {DATA} --out {CACHE}
        !tar -cf "{TAR}" -C {CACHE} .
!ls {CACHE} | wc -l

## 4 · Experiments
Each run saves to Drive and can be resumed after a disconnect with `--resume`.

**Step 4a — illumination ablation (P3)** on the baseline: none vs Graham vs CLAHE. Pick the best by **validation** QWK.

In [ ]:
def train(cfg, name=None, extra=""):
    """Run one experiment; output streams live into the cell and is also saved to train.log."""
    name = name or cfg
    get_ipython().system(f'python scripts/train.py --config configs/{cfg}.yaml --splits data/splits.csv '
                         f'--images {CACHE} --out "{RUNS}/{name}" --resume {extra}')

train("A_baseline")
train("A_graham")
train("A_clahe")

In [ ]:
import pandas as pd, json, glob
def summary():
    rows = []
    for d in sorted(glob.glob(f"{RUNS}/*/val_metrics.json")):
        m = json.load(open(d)); name = d.split("/")[-2]
        rows.append({"run": name, "QWK": m["qwk"], "macro-F1": m["macro_f1"], "accuracy": m["accuracy"],
                     "ECE": m["ece"], "under-grading": m["under_grading_rate"],
                     "referable sens": m["referable"]["sensitivity"], "referable spec": m["referable"]["specificity"]})
    return pd.DataFrame(rows).set_index("run").round(4)
summary()

**Step 4b — ladder B → C → D** with the winning normaliser (one change at a time; A with the same normaliser is the baseline).

In [ ]:
NORM = "none"     # ← set to the winner of step 4a: none | graham | clahe
for cfg in ["B_ordinal", "C_multitask", "D_bottleneck"]:
    train(cfg, f"{cfg}_{NORM}", f"data.normaliser={NORM}")
summary()

## 5 · Evaluation on the held-out test set (run once per model)
Model selection, temperature, decision rule, operating thresholds and conformal thresholds were all fixed on
**validation**. The test set is scored once for the final ladder (A → B → C → D); the illumination-ablation
runs are not evaluated on test because they were only used for a validation decision.

In [ ]:
FINAL = ["A_baseline", "B_ordinal_none", "C_multitask_none", "D_bottleneck_none"]
MAIN  = "C_multitask_none"          # best validation QWK
for r in FINAL:
    get_ipython().system(f'python scripts/evaluate.py --run "{RUNS}/{r}" --splits data/splits.csv --images {CACHE}')

In [ ]:
def test_summary():
    rows = []
    for r in FINAL:
        t = json.load(open(f"{RUNS}/{r}/eval/test_metrics.json"))["test"]; c = t["ci95"]
        f = lambda k: f"{c[k][0]:.3f} [{c[k][1]:.3f}, {c[k][2]:.3f}]"
        rows.append({"run": r, "QWK [95% CI]": f("qwk"), "macro-F1": f("macro_f1"), "accuracy": f("accuracy"),
                     "referable sens": f("referable_sensitivity"), "referable spec": f("referable_specificity"),
                     "STDR sens": f("sight_threatening_sensitivity"), "under-grading": round(t["under_grading_rate"], 3),
                     "distant errors": round(t["distant_error_rate"], 3), "missed STDR (≤ Mild)": t["missed_sight_threatening"],
                     "ECE raw→cal": f"{t['ece_uncalibrated']:.3f}→{t['ece_calibrated']:.3f}",
                     "90% set coverage": round(t["conformal"]["coverage"], 3), "mean set size": round(t["conformal"]["mean_set_size"], 2),
                     "flagged": f"{t['review_flags']['any_flag']['rate']:.1%}"})
    return pd.DataFrame(rows).set_index("run")
ts = test_summary(); ts.to_csv(f"{RUNS}/test_summary.csv"); ts

## 6 · Robustness (baseline vs main model) and explanations

In [ ]:
for r in ["A_baseline", MAIN]:
    get_ipython().system(f'python scripts/robustness.py --run "{RUNS}/{r}" --splits data/splits.csv --images {CACHE}')
for r in [MAIN, "D_bottleneck_none"]:
    get_ipython().system(f'python scripts/explain.py --run "{RUNS}/{r}" --splits data/splits.csv --images {CACHE}')

In [ ]:
from IPython.display import Image, display
for f in ["eval/confusion_matrix.png", "eval/reliability.png", "eval/selective_prediction.png",
          "robustness/robustness.png", "explain/gallery.png", "explain/sanity_check.png"]:
    print(f); display(Image(f"{RUNS}/{MAIN}/{f}", width=900))